# Lab 01: Data Pipeline

## Business Context

The firm has 25 S-1 filings from recent tech IPOs sitting in a Unity Catalog Volume, and stock return data scattered across Yahoo Finance. Before the analyzer can answer any questions, both data sources need to be in one place — parsed, chunked, indexed, and queryable.

**After this lab:** All filings are parsed into searchable chunks in a Delta table with a Vector Search index, and stock performance data is loaded into a structured Delta table. The foundation for everything that follows.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Data Preparation (14%) |
| **Key Skills** | ai_parse_document, chunking, Delta Sync, managed embeddings, structured data ingestion |
| **Estimated Time** | 40 minutes |
| **Estimated Cost** | ~$3-4 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-text-splitters yfinance --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/sec_filings"
LLM_ENDPOINT = "databricks-llama-4-maverick"
VS_ENDPOINT = "ipo_analyzer_vs_endpoint"

print(f"Catalog : {CATALOG}")
print(f"Volume  : {VOLUME_PATH}")

# List available filings
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

## A. Parse S-1 Filings

`ai_parse_document()` extracts structured content (text, tables, images) from documents. It returns a **VARIANT** type — a semi-structured JSON column that SQL path expressions navigate naturally.

> **Why SQL for VARIANT?** SQL path expressions like `parsed_content:document:elements[0]:content::STRING` are the native, concise way to access semi-structured data. PySpark requires `from_json()` with explicit schemas — more verbose and fragile when the structure varies between documents.

In [ ]:
from pyspark.sql.functions import expr

# Read all filings as binary files
docs_df = spark.read.format("binaryFile").load(VOLUME_PATH)
print(f"Found {docs_df.count()} filings")

# Parse each document
parsed_df = docs_df.withColumn(
    "parsed_content",
    expr("ai_parse_document(content, map('version', '2.0'))")
)

# Save to table (VARIANT type needs SQL for best compat)
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.parsed_filings"
parsed_df.select("path", "parsed_content").write.mode("overwrite").saveAsTable(PARSED_TABLE)
print(f"Parsed filings saved to {PARSED_TABLE}")

# Show parsing results
display(spark.sql(f"""
SELECT
  path,
  parsed_content:error_status::STRING AS error_status
FROM {PARSED_TABLE}
"""))

In [ ]:
# Text lives in elements[*].content, NOT pages[*].text
# (pages only have id + image_uri in ai_parse_document v2.0)
elements_df = spark.sql(f"""
SELECT
  path,
  elem.content::STRING AS element_text
FROM {CATALOG}.{SCHEMA}.parsed_filings
LATERAL VIEW explode(from_json(
  parsed_content:document:elements::STRING,
  'ARRAY<STRUCT<content: STRING>>'
)) AS elem
WHERE elem.content IS NOT NULL
""")

elements_df.createOrReplaceTempView("elements")
print(f"Extracted {elements_df.count()} text elements across all filings")
display(elements_df.limit(5))

# ── Python alternative (equivalent, more verbose): ────────────────────────
# from pyspark.sql.functions import explode, from_json, col
# from pyspark.sql.types import ArrayType, StructType, StructField, StringType
#
# schema = ArrayType(StructType([StructField("content", StringType())]))
# elements_df = (
#     spark.table(f"{CATALOG}.{SCHEMA}.parsed_filings")
#     .withColumn("elements", from_json(col("parsed_content:document:elements").cast("string"), schema))
#     .withColumn("elem", explode("elements"))
#     .select("path", col("elem.content").alias("element_text"))
#     .filter(col("element_text").isNotNull())
# )

## B. Load Stock Performance Data

For each company, we download post-IPO stock prices from Yahoo Finance and calculate 3, 6, and 12-month returns. This structured data lives in a Delta table alongside the unstructured S-1 text — the kind of cross-data join that no chat tool can do at scale.

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# Company list — same source as the setup script
COMPANIES = [
    {"company": "Snowflake",    "ticker": "SNOW",  "ipo_date": "2020-09-16", "sector": "Cloud/Data"},
    {"company": "Palantir",     "ticker": "PLTR",  "ipo_date": "2020-09-30", "sector": "Data Analytics"},
    {"company": "DoorDash",     "ticker": "DASH",  "ipo_date": "2020-12-09", "sector": "Delivery"},
    {"company": "Coinbase",     "ticker": "COIN",  "ipo_date": "2021-04-14", "sector": "Crypto/Fintech"},
    {"company": "Rivian",       "ticker": "RIVN",  "ipo_date": "2021-11-10", "sector": "EV/Auto"},
    {"company": "Unity",        "ticker": "U",     "ipo_date": "2020-09-18", "sector": "Gaming/Dev Tools"},
    {"company": "Roblox",       "ticker": "RBLX",  "ipo_date": "2021-03-10", "sector": "Gaming"},
    {"company": "Bumble",       "ticker": "BMBL",  "ipo_date": "2021-02-11", "sector": "Social/Dating"},
    {"company": "Affirm",       "ticker": "AFRM",  "ipo_date": "2021-01-13", "sector": "Fintech"},
    {"company": "Robinhood",    "ticker": "HOOD",  "ipo_date": "2021-07-29", "sector": "Fintech"},
    {"company": "Toast",        "ticker": "TOST",  "ipo_date": "2021-09-22", "sector": "Restaurant Tech"},
    {"company": "Confluent",    "ticker": "CFLT",  "ipo_date": "2021-06-24", "sector": "Data Streaming"},
    {"company": "GitLab",       "ticker": "GTLB",  "ipo_date": "2021-10-14", "sector": "DevOps"},
    {"company": "HashiCorp",    "ticker": "HCP",   "ipo_date": "2021-12-09", "sector": "Cloud Infra"},
    {"company": "Duolingo",     "ticker": "DUOL",  "ipo_date": "2021-07-28", "sector": "EdTech"},
    {"company": "Instacart",    "ticker": "CART",  "ipo_date": "2023-09-19", "sector": "Delivery"},
    {"company": "Klaviyo",      "ticker": "KVYO",  "ipo_date": "2023-09-20", "sector": "Marketing Tech"},
    {"company": "Arm Holdings", "ticker": "ARM",   "ipo_date": "2023-09-14", "sector": "Semiconductors"},
    {"company": "Reddit",       "ticker": "RDDT",  "ipo_date": "2024-03-21", "sector": "Social Media"},
    {"company": "Rubrik",       "ticker": "RBRK",  "ipo_date": "2024-04-25", "sector": "Cybersecurity"},
    {"company": "Astera Labs",  "ticker": "ALAB",  "ipo_date": "2024-03-20", "sector": "Semiconductors"},
    {"company": "Ibotta",       "ticker": "IBTA",  "ipo_date": "2024-04-18", "sector": "Fintech"},
]

rows = []
for c in COMPANIES:
    ticker = c["ticker"]
    ipo_date = c["ipo_date"]
    print(f"  {ticker}: fetching post-IPO prices...")
    try:
        stock = yf.Ticker(ticker)
        # Get 14 months of data starting from IPO date
        end_date = (datetime.strptime(ipo_date, "%Y-%m-%d") + timedelta(days=420)).strftime("%Y-%m-%d")
        hist = stock.history(start=ipo_date, end=end_date)

        if len(hist) < 2:
            print(f"    WARNING: insufficient data for {ticker}")
            continue

        ipo_price = hist.iloc[0]["Close"]

        # Find prices at 3, 6, 12 months post-IPO
        ipo_dt = pd.Timestamp(ipo_date)
        dates_3m = hist.index[hist.index >= ipo_dt + pd.DateOffset(months=3)]
        dates_6m = hist.index[hist.index >= ipo_dt + pd.DateOffset(months=6)]
        dates_12m = hist.index[hist.index >= ipo_dt + pd.DateOffset(months=12)]

        price_3m = hist.loc[dates_3m[0]]["Close"] if len(dates_3m) > 0 else None
        price_6m = hist.loc[dates_6m[0]]["Close"] if len(dates_6m) > 0 else None
        price_12m = hist.loc[dates_12m[0]]["Close"] if len(dates_12m) > 0 else None

        return_12m = round(((price_12m - ipo_price) / ipo_price * 100), 1) if price_12m else None

        rows.append({
            "company": c["company"],
            "ticker": ticker,
            "sector": c["sector"],
            "ipo_date": ipo_date,
            "ipo_price": round(float(ipo_price), 2),
            "price_3m": round(float(price_3m), 2) if price_3m else None,
            "price_6m": round(float(price_6m), 2) if price_6m else None,
            "price_12m": round(float(price_12m), 2) if price_12m else None,
            "twelve_month_return_pct": float(return_12m) if return_12m else None,
        })
    except Exception as e:
        print(f"    ERROR for {ticker}: {e}")

stock_pdf = pd.DataFrame(rows)
stock_df = spark.createDataFrame(stock_pdf)
STOCK_TABLE = f"{CATALOG}.{SCHEMA}.stock_performance"
stock_df.write.mode("overwrite").saveAsTable(STOCK_TABLE)
print(f"\nSaved {len(rows)} companies to {STOCK_TABLE}")

In [ ]:
display(spark.sql(f"""
SELECT company, ticker, sector, ipo_date, ipo_price, price_12m, twelve_month_return_pct
FROM {CATALOG}.{SCHEMA}.stock_performance
ORDER BY twelve_month_return_pct DESC
"""))

## C. Chunk Filing Text

We split the extracted text into chunks suitable for embedding and vector search. Key parameters:
- **chunk_size = 1000**: Maximum characters per chunk
- **chunk_overlap = 200**: Characters shared between adjacent chunks (prevents context loss at boundaries)

Recursive character splitting tries paragraph boundaries first, then sentences, then words.

In [ ]:
from pyspark.sql.functions import concat_ws, collect_list
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Concatenate all element text per filing
raw_text_df = elements_df.groupBy("path").agg(
    concat_ws("\n\n", collect_list("element_text")).alias("raw_text")
)

# Chunk using RecursiveCharacterTextSplitter
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Collect to driver for chunking (dataset is small enough — 22 filings)
raw_rows = raw_text_df.collect()

chunks = []
for row in raw_rows:
    text_chunks = text_splitter.split_text(row.raw_text) if row.raw_text else []
    for i, chunk in enumerate(text_chunks):
        chunks.append((row.path, i, chunk))

schema = StructType([
    StructField("path", StringType()),
    StructField("chunk_index", IntegerType()),
    StructField("chunk_text", StringType()),
])
chunks_df = spark.createDataFrame(chunks, schema)

# Add unique chunk ID (required for Vector Search)
chunks_flat_df = chunks_df.withColumn(
    "chunk_id",
    F.sha2(F.concat(F.col("path"), F.lit("_"), F.col("chunk_index").cast("string")), 256)
).select("chunk_id", "path", "chunk_index", "chunk_text")

print(f"Total chunks: {chunks_flat_df.count()}")

In [ ]:
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.filing_chunks"

(
    chunks_flat_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(CHUNKS_TABLE)
)

print(f"Saved to {CHUNKS_TABLE}")

# Show chunks per filing
display(spark.sql(f"""
SELECT
  SUBSTRING_INDEX(path, '/', -1) AS filing,
  COUNT(*) AS chunks,
  ROUND(AVG(LENGTH(chunk_text))) AS avg_chunk_len
FROM {CHUNKS_TABLE}
GROUP BY path
ORDER BY chunks DESC
"""))

## D. Create Vector Search Index

A **Vector Search Endpoint** hosts the compute. A **Delta Sync Index** with **Managed Embeddings** auto-generates embeddings using `databricks-bge-large-en` and stays in sync with the source Delta table via Change Data Feed.

Prerequisites (already done):
1. Source table has a unique primary key (`chunk_id`) ✓
2. Source table has Change Data Feed enabled ✓

In [ ]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

# Create or reuse existing endpoint
try:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Creating endpoint '{VS_ENDPOINT}'...")
except Exception as e:
    if "already exists" in str(e).lower() or "quota" in str(e).lower():
        print(f"Endpoint '{VS_ENDPOINT}' already exists, reusing.")
    else:
        raise

# Wait for endpoint to be online
for i in range(30):
    try:
        ep = vsc.get_endpoint(VS_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"  Endpoint status: {status}")
        if status == "ONLINE":
            break
    except Exception:
        pass
    time.sleep(10)

print(f"Endpoint '{VS_ENDPOINT}' ready.")

In [ ]:
# Create Delta Sync index with managed embeddings
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=f"{CATALOG}.{SCHEMA}.filing_chunks",
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"Creating index '{VS_INDEX}'...")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{VS_INDEX}' already exists, reusing.")
    else:
        raise

# Wait for index to sync
print("Waiting for index to sync (may take several minutes)...")
for i in range(60):
    try:
        idx = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        status = idx.describe().get("status", {})
        ready = status.get("ready", False)
        msg = status.get("message", "")
        print(f"  Index: ready={ready} | {msg}")
        if ready:
            break
    except Exception as e:
        print(f"  Checking: {e}")
    time.sleep(15)

print(f"Index '{VS_INDEX}' ready.")

In [ ]:
# Test: semantic search for "competition"
idx = vsc.get_index(VS_ENDPOINT, VS_INDEX)
results = idx.similarity_search(
    query_text="competitive landscape and market position",
    columns=["chunk_id", "path", "chunk_text"],
    num_results=3,
    query_type="HYBRID",
)

print("=== Hybrid Search Results ===")
for doc in results.get("result", {}).get("data_array", []):
    source = doc[1].split("/")[-1].replace("-S1.html", "")
    print(f"\nSource: {source}")
    print(f"  {doc[2][:200]}...")

## Before / After

**Before this lab:** S-1 filings are raw HTML files in a volume. Stock data is on Yahoo Finance. To find "what did Snowflake say about competition?", you'd open each filing manually and Ctrl+F.

**After this lab:** Both data sources are in Delta tables. You can find relevant passages in milliseconds via Vector Search, and cross-reference with stock performance data.

In [ ]:
# BEFORE: SQL LIKE scan — slow, misses synonyms
print("=== BEFORE: SQL LIKE scan ===")
like_results = spark.sql(f"""
SELECT path, chunk_text
FROM {CATALOG}.{SCHEMA}.filing_chunks
WHERE LOWER(chunk_text) LIKE '%competitive%'
LIMIT 3
""")
print(f"SQL LIKE 'competitive': {like_results.count()} results (exact match only)")
display(like_results.select("path", F.substring("chunk_text", 1, 150).alias("preview")))

print()

# AFTER: Vector Search — fast, semantic
print("=== AFTER: Vector Search (semantic) ===")
vs_results = idx.similarity_search(
    query_text="competitive landscape and market position",
    columns=["path", "chunk_text"],
    num_results=3,
    query_type="HYBRID",
)
vs_docs = vs_results.get("result", {}).get("data_array", [])
print(f"Vector Search 'competitive landscape': {len(vs_docs)} results (semantic match)")
for doc in vs_docs:
    source = doc[0].split("/")[-1].replace("-S1.html", "")
    print(f"  [{source}] {doc[1][:150]}...")

print()
print("Stock data preview:")
display(spark.sql(f"""
SELECT company, ticker, twelve_month_return_pct
FROM {CATALOG}.{SCHEMA}.stock_performance
ORDER BY twelve_month_return_pct DESC
LIMIT 5
"""))

---

## Exam Preparation

### Key Concepts

| Concept | Definition |
|---|---|
| `ai_parse_document()` | Databricks SQL function that extracts text, tables, and images from documents using Mosaic AI; returns VARIANT type |
| VARIANT | Semi-structured data type in Databricks; accessed via SQL path expressions like `col:field::TYPE` |
| Recursive Character Splitting | Chunking strategy that tries to split on semantic boundaries (paragraphs > sentences > words) |
| Chunk Overlap | Shared text between adjacent chunks to prevent context loss at boundaries |
| Change Data Feed (CDF) | Delta Lake feature that tracks row-level changes; required for Delta Sync Vector Search |
| Delta Sync Index | Vector Search index that auto-syncs with a source Delta table using CDF |
| Managed Embeddings | Databricks generates and stores embeddings using a specified model endpoint; no user code needed |
| Hybrid Search | Combines semantic (embedding) + keyword (BM25) results via Reciprocal Rank Fusion (RRF) |

### Practice Questions

**Q1.** What are the prerequisites for creating a Delta Sync Vector Search index?

- A) The source table must use Parquet format
- B) The source table must have a unique primary key and Change Data Feed enabled
- C) The source table must be in the hive_metastore catalog
- D) The source table must have pre-computed embeddings

**Answer: B** — Delta Sync requires a unique key for tracking changes and CDF for incremental sync.

---

**Q2.** Why does `ai_parse_document()` return a VARIANT type rather than a structured schema?

- A) VARIANT is faster to query than structured columns
- B) Document structure varies between files — VARIANT handles schema variability naturally
- C) VARIANT uses less storage than structured types
- D) Structured schemas are not supported in Unity Catalog

**Answer: B** — Different documents have different structures (number of pages, element types, metadata). VARIANT accommodates this without requiring a fixed schema.

---

**Q3.** When should you use hybrid search instead of pure semantic search?

- A) When you need the fastest possible query time
- B) When queries mix natural language with specific technical terms
- C) When the index has fewer than 100 documents
- D) When you want to minimize token usage

**Answer: B** — Hybrid combines semantic recall (catches meaning) with keyword precision (catches exact terms). Recommended default for production RAG.

---

**Q4.** What is the purpose of chunk overlap?

- A) To reduce the total number of chunks
- B) To ensure every chunk has the same length
- C) To prevent context from being lost at chunk boundaries
- D) To improve embedding model accuracy

**Answer: C** — Content near chunk boundaries might be split across two chunks. Overlap ensures important context spanning a boundary appears in both chunks.

---

**Q5.** A vector database supports 100M embeddings but chunking produces 150M. How should you reduce chunk count?

- A) Use a larger embedding model
- B) Increase chunk_size to produce fewer, larger chunks
- C) Remove chunk overlap entirely
- D) Switch from Delta Sync to manual indexing

**Answer: B** — Larger chunks mean fewer total chunks per document, reducing the total embedding count.

### Cost Breakdown

| Component | Detail | Est. Cost |
|---|---|---|
| Serverless compute | Parsing + chunking + stock data (~30 min) | ~$1.00 |
| Managed embeddings | Embedding all chunks on index creation | ~$0.50 |
| VS endpoint | Running during lab (~30 min) | ~$1.00 |
| yfinance API | Free | $0 |
| **Total** | | **~$3-4** |